# Stage 0: Data Inspection and Cleaning
This notebook loads, inspects, cleans, and profiles the NVDA dataset.


In [ ]:
# ============================================================
# TICKER PARAMETER — change this to run for any stock
# ============================================================
TICKER = 'NVDA'   # Options: AAPL AMZN BAC GOOGL JNJ JPM MSFT NVDA UNH XOM

import os, sys
sys.path.insert(0, os.path.abspath('../..'))

DATA_RAW     = f'../../data/raw/{TICKER}_raw.csv'
DATA_CLEAN   = f'../../data/processed/per_stock/{TICKER}_clean.parquet'
DATA_CUSUM   = f'../../data/processed/per_stock/{TICKER}_cusum_events.parquet'
DATA_BARS    = f'../../data/processed/per_stock/{TICKER}_dollar_bars.parquet'
DATA_LABELS  = f'../../data/processed/per_stock/{TICKER}_labels.parquet'
DATA_WEIGHTS = f'../../data/processed/per_stock/{TICKER}_weights.parquet'
DATA_FRACDIFF= f'../../data/processed/per_stock/{TICKER}_fracdiff.parquet'
DATA_FEATURES= f'../../data/processed/per_stock/{TICKER}_ts_features.parquet'
DATA_BSADF   = f'../../data/processed/per_stock/{TICKER}_bsadf.parquet'
FIG_DIR      = f'../../reports/figures/per_stock/{TICKER}'
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs('../../data/processed/per_stock', exist_ok=True)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import os

# Ensure output directories exist
os.makedirs(os.path.dirname(DATA_CLEAN), exist_ok=True)
plt.style.use('seaborn-v0_8-darkgrid')


## 1. Load Data


In [ ]:
def load_and_clean(path):
    df = pd.read_csv(path)
    df['Date'] = pd.to_datetime(df['Date'])
    df.set_index('Date', inplace=True)
    df.sort_index(inplace=True)
    return df

file_path = DATA_RAW
df = load_and_clean(file_path)
df.head()


## 2. Compute returns and dollar volume


In [ ]:
def compute_dollar_volume(df):
    return df['Adj Close'] * df['Volume']

df['log_return'] = np.log(df['Adj Close'] / df['Adj Close'].shift(1))
df['dollar_volume'] = compute_dollar_volume(df)
df.dropna(inplace=True)
df.head()


## 3. Basic Statistics and Validation


In [ ]:
print(f"Shape: {df.shape}")
print(f"Date Range: {df.index.min()} to {df.index.max()}")
print(f"Missing Values:\n{df.isnull().sum()}")
print("\nBasic Statistics:")
display(df.describe())

# Validation checks
assert df.isnull().sum().sum() == 0, "NaN values found!"
assert df.index.is_monotonic_increasing, "Dates are not monotonic!"
assert (df[['Adj Close', 'Close', 'High', 'Low', 'Open']] > 0).all().all(), "Prices must be > 0"
assert (df['Volume'] > 0).all(), "Volume must be > 0"


## 4. Detect Outliers


In [ ]:
def detect_outliers(returns, sigma=5):
    mean_ret = returns.mean()
    std_ret = returns.std()
    return np.abs(returns - mean_ret) > (sigma * std_ret)

outlier_mask = detect_outliers(df['log_return'])
outliers = df[outlier_mask]
print(f"Found {len(outliers)} outliers:")
display(outliers[['log_return']])


## 5. Plots


In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(15, 18))

# a. NVDA Adj Close price chart (log scale)
axes[0, 0].plot(df.index, df['Adj Close'], color='blue')
axes[0, 0].set_yscale('log')
axes[0, 0].set_title('NVDA Adj Close Price (Log Scale)')
axes[0, 0].set_ylabel('Price')

# b. Daily log return distribution with Jarque-Bera
jb_stat, jb_p = stats.jarque_bera(df['log_return'])
axes[0, 1].hist(df['log_return'], bins=100, alpha=0.7, color='green', density=True)
axes[0, 1].set_title(f'Log Return Distribution\nJarque-Bera: stat={jb_stat:.2f}, p={jb_p:.4f}')
axes[0, 1].set_xlabel('Log Return')

# c. QQ plot
stats.probplot(df['log_return'], dist="norm", plot=axes[1, 0])
axes[1, 0].set_title('QQ Plot of Daily Returns')

# d. Rolling 60-day volatility (annualised)
roll_vol = df['log_return'].rolling(window=60).std() * np.sqrt(252)
axes[1, 1].plot(df.index, roll_vol, color='red')
axes[1, 1].set_title('Rolling 60-day Volatility (Annualised)')
axes[1, 1].set_ylabel('Volatility')

# e. Dollar volume time series
axes[2, 0].plot(df.index, df['dollar_volume'], color='purple')
axes[2, 0].set_title('Daily Dollar Volume')
axes[2, 0].set_ylabel('Dollar Volume')

axes[2, 1].axis('off') # Hide unused subplot
plt.tight_layout()
plt.show()


## 6. Save Cleaned Data


In [ ]:
df.to_parquet(f'../../data/processed/per_stock/{TICKER}_clean.parquet')
print(f"Saved {TICKER} data to {DATA_CLEAN}")
